# 5G Network Intelligence — Telecom Experiment Integrity and Replay

## Verification question
Do the three completed data/compute arms satisfy their declared temporal, seed, transfer, compute and scoring contracts? This notebook performs inference and verification only. It never trains, calibrates, selects new weights or opens original hours 504 onward.


In [ ]:
# Default Run All is a report review.
RUN_AUDIT = False


## Verified input provenance
Use the original hosted source prefix and audited Telecom supplement. Validate every manifest member before importing code. Completed checkpoints remain in attached Kaggle Input; only aggregate verification/metrics are exported. The training source stays pinned to its original commit.


In [ ]:
from pathlib import Path, PurePosixPath
import os, sys, json, hashlib, shutil
PROJECT = Path.cwd()
if not (PROJECT/'configs').is_dir():
    PROJECT = PROJECT.parent
def verified_input(filename, digest):
    matches = [p for p in Path('/kaggle/input').rglob(filename)
               if hashlib.sha256(p.read_bytes()).hexdigest() == digest]
    if len(matches) != 1:
        raise ValueError('Exactly one pinned input manifest required')
    source = matches[0]
    manifest = json.loads(source.read_text())
    for name, expected in manifest['files'].items():
        relative = PurePosixPath(name)
        if relative.is_absolute() or '..' in relative.parts or ':' in name or '\\' in name:
            raise ValueError('Unsafe input path')
        path = source.parent/name
        if not path.resolve().is_relative_to(source.parent.resolve()) or path.is_symlink():
            raise ValueError('Input path escapes manifest root')
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise ValueError('Input checksum differs: '+name)
    return source.parent, manifest


In [ ]:
if RUN_AUDIT:
    assert Path('/kaggle/working').is_dir(), 'Hosted Kaggle worker only.'
    base_root, base = verified_input('source_manifest.json', '3069c57c9f64f809464ead2fae973fbaa35a86c3a46884e15cc693354eecab46')
    extra_root, extra = verified_input('telecom_transfer_manifest.json', '10cd970e8724ca5258b855a37e2b3ebbda41b6e80b9984ad83cb56335ba702ed')
    assert not extra['original_beam_payload_included']
    PROJECT = Path('/kaggle/working/telecom_transfer_audit')
    PROJECT.mkdir(exist_ok=True)
    # Only reuse base data/reports; import all code from the new pinned supplement.
    for source_root, manifest, base_only in [(base_root, base, True), (extra_root, extra, False)]:
        for name in manifest['files']:
            if base_only and not name.startswith(('data/', 'reports/')):
                continue
            target = PROJECT/name
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(source_root/name, target)
    os.chdir(PROJECT)
    sys.path.insert(0, str(PROJECT))
    os.environ['SOURCE_GIT_COMMIT'] = extra['source_commit']
    print('Verified source:', extra['source_commit'])


## Independent checks
Verify exact artifact coverage/hashes; reconstruct fit-prefix preprocessing; reproduce source and target cold weights from seed 17; compare target cold states bitwise across arms. Check 5,120/15,360 actual optimizer updates and encoder-only transfer, unchanged target heads/scales, completed stopping, all original target/origin/beam axes, full-population MAE/WAPE, and CPU/GPU replay at atol 0.0001. No tolerance is selected from the current scores.


In [ ]:
if RUN_AUDIT:
    import urllib.request, subprocess
    # Public code only: immutable commit plus exact content hash; no credentials.
    with urllib.request.urlopen('https://raw.githubusercontent.com/trbk07/5g-network-intelligence/bc454cd3a3a60335cf90dc36e47008b72f0e4c4a/scripts/audit_telecom_cloud.py', timeout=60) as response:
        code = response.read()
    assert hashlib.sha256(code).hexdigest() == 'cc9999bc7e52eecbc109e3eb95069cc0ad1a99cc9fd5cc3c7a57a60eb2ec8b92', 'Audit source checksum differs'
    audit_script = PROJECT/'audit_telecom_cloud.py'
    audit_script.write_bytes(code)
    matches = [p.parent for p in Path('/kaggle/input').rglob('manifest.json')
               if p.parent.name == 'telecom_transfer' and all((p.parent/arm).is_dir()
               for arm in ['telecom_matched', 'existing_extended', 'telecom_extended'])]
    assert len(matches) == 1, 'Completed training artifact tree must be uniquely attached'
    subprocess.run([sys.executable, str(audit_script), '--artifacts', str(matches[0]),
        '--source-commit', '63187481ba9c7dfe2f3f56375aca0e30993298ee'], check=True)
else:
    print('Review mode: no checkpoint access, fitting or inference.')


## Metric review and limitations
MAE uses original DLPRB usage units; WAPE is total absolute error over total absolute observed load. Numerical parity is an engineering check, not a generalization claim. All arms use repeatedly reused development. Partial source overlap and original calendar identity remain uncertified. Quality gates and paired comparisons with the fixed strong comparator follow verification; a successful audit alone does not establish improvement, WAPE below 10% or production readiness.


In [ ]:
# Read only completed aggregate evidence.
import pandas as pd
from IPython.display import display
path = Path('/kaggle/working/telecom_transfer_verified_metrics.json') if RUN_AUDIT else PROJECT/'reports/telecom_transfer_verified_metrics.json'
if path.exists():
    result = json.loads(path.read_text(encoding='utf-8'))
    assert result['all_metrics_reconstructed'] and not result['holdout_used']
    frame = pd.DataFrame(result['metrics'])
    frame['WAPE (%)'] = 100*frame['wape']
    display(frame)
    display({k: v for k, v in result.items() if k != 'metrics'})
else:
    print('Verification pending; no accepted quality result yet.')
